In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
SupportsFloat = float
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

NANX_RESULTS_DIR = Path("/tmp/plaintext_sims_results")
NANX_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

_NANX_RESULTS_ROWS = {
    "total_calendar_time": [114.9, 103.0],
    "num_rework_cycles": [0.6, 1.2],
    "prop_defects_caught_early": [0.82, 0.73],
    "num_late_defects": [0.3, 0.9],
    "handover_delay": [0.5, 2.2],
    "condition": ["plaintext", "mixed"],
    "seed": [279223610, 423198623],
}
_NANX_BOOT_ROWS = [
    {"metric": "total_calendar_time", "delta_mean": 11.868794921705613, "ci_lower": 11.371132533828291, "ci_upper": 12.387187407199493},
    {"metric": "num_rework_cycles", "delta_mean": -0.5644594, "ci_lower": -0.5884025, "ci_upper": -0.5410000000000001},
]

# --- run_concat_write ---
FIX_RUN_CONCAT_WRITE_RESULTS_DIR = NANX_RESULTS_DIR
FIX_RUN_CONCAT_WRITE_SIM_MIXED_BEFORE = pd.DataFrame(_NANX_RESULTS_ROWS).query("condition == 'mixed'").reset_index(drop=True)
FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_BEFORE = pd.DataFrame(_NANX_RESULTS_ROWS).query("condition == 'plaintext'").reset_index(drop=True)
FIX_RUN_CONCAT_WRITE_SIM_MIXED_GEN = pl.from_pandas(FIX_RUN_CONCAT_WRITE_SIM_MIXED_BEFORE)
FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_GEN = pl.from_pandas(FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_BEFORE)

# --- run_dataframe_write ---
FIX_RUN_DATAFRAME_WRITE_RESULTS_DIR = NANX_RESULTS_DIR
FIX_RUN_DATAFRAME_WRITE_SIM_BOOT = _NANX_BOOT_ROWS

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_run_concat_write(results_dir, sim_mixed, sim_plaintext):
    sim_results = pd.concat([sim_plaintext, sim_mixed], ignore_index=True)
    sim_results.to_csv(results_dir / "simpy_results.csv", index=False)
    return sim_results

def before_run_dataframe_write(results_dir, sim_boot):
    pd.DataFrame(sim_boot).to_csv(
        results_dir / "simpy_bootstrap_effects.csv", index=False
    )
    return None

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_run_concat_write(results_dir, sim_mixed, sim_plaintext):

    sim_results = pl.concat([sim_plaintext, sim_mixed], how="vertical")
    sim_results.write_csv(results_dir / "simpy_results.csv")
    return sim_results

def gen_run_dataframe_write(results_dir, sim_boot):

    pl.DataFrame(sim_boot).write_csv(results_dir / "simpy_bootstrap_effects.csv")
    return None

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: run_concat_write ===

# L1 smoke – generated
try:
    _r = gen_run_concat_write(FIX_RUN_CONCAT_WRITE_RESULTS_DIR, FIX_RUN_CONCAT_WRITE_SIM_MIXED_GEN, FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_GEN)
    print("✅ L1 smoke gen_run_concat_write: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_run_concat_write: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_run_concat_write(FIX_RUN_CONCAT_WRITE_RESULTS_DIR, FIX_RUN_CONCAT_WRITE_SIM_MIXED_BEFORE, FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_BEFORE)
    print("✅ L1 smoke before_run_concat_write: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_run_concat_write: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_run_concat_write(FIX_RUN_CONCAT_WRITE_RESULTS_DIR, FIX_RUN_CONCAT_WRITE_SIM_MIXED_BEFORE, FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_BEFORE)
    _rg = gen_run_concat_write(FIX_RUN_CONCAT_WRITE_RESULTS_DIR, FIX_RUN_CONCAT_WRITE_SIM_MIXED_GEN, FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_GEN)
    compare(_rb, _rg, "run_concat_write")
except Exception as _e:
    print(f"❌ L2 equivalence run_concat_write: setup error — {type(_e).__name__}: {_e}")

# L3 edge - compare both returned frames and written CSV files.
import tempfile
try:
    with tempfile.TemporaryDirectory() as _bd, tempfile.TemporaryDirectory() as _gd:
        _bdir, _gdir = Path(_bd), Path(_gd)
        _rb = before_run_concat_write(_bdir, FIX_RUN_CONCAT_WRITE_SIM_MIXED_BEFORE.head(0), FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_BEFORE.head(0))
        _rg = gen_run_concat_write(_gdir, FIX_RUN_CONCAT_WRITE_SIM_MIXED_GEN.head(0), FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_GEN.head(0))
        compare(_rb, _rg, "L3 edge run_concat_write empty return", check_row_order=True)
        _bf = pd.read_csv(_bdir / "simpy_results.csv")
        _gf = pl.read_csv(_gdir / "simpy_results.csv")
        compare(_bf, _gf, "L3 edge run_concat_write empty CSV", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge run_concat_write: {type(_e).__name__}: {_e}")

# AUDIT-127-L2: verify the standard-path CSV side effect too.
import tempfile
try:
    with tempfile.TemporaryDirectory() as _bd, tempfile.TemporaryDirectory() as _gd:
        _bdir, _gdir = Path(_bd), Path(_gd)
        before_run_concat_write(_bdir, FIX_RUN_CONCAT_WRITE_SIM_MIXED_BEFORE, FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_BEFORE)
        gen_run_concat_write(_gdir, FIX_RUN_CONCAT_WRITE_SIM_MIXED_GEN, FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_GEN)
        compare(pd.read_csv(_bdir / "simpy_results.csv"), pl.read_csv(_gdir / "simpy_results.csv"), "run_concat_write CSV side effect", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence run_concat_write CSV side effect: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_run_concat_write: OK, type= DataFrame
✅ L1 smoke before_run_concat_write: OK
✅ L2 equivalence run_concat_write: MATCH
✅ L3 edge run_concat_write empty return: MATCH
✅ L3 edge run_concat_write empty CSV: MATCH
✅ L2 equivalence run_concat_write CSV side effect: MATCH
